<a href="https://colab.research.google.com/github/solen16/Satisfaction_dev/blob/main/ML_Camembert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# chargement et concaténation de plusieurs fichiers csv
import pandas as pd
import glob

# 1. Indiquez le chemin vers le dossier contenant vos 16 fichiers CSV
# (par exemple un dossier nommé 'data_cv')
chemin_dossier = "extraction_supabase/*.csv"

# 2. Récupérer la liste de tous les fichiers CSV
fichiers_csv = glob.glob(chemin_dossier)

print(f"{len(fichiers_csv)} fichiers trouvés.")

# 3. Lire et concaténer tous les DataFrames
liste_dfs = [pd.read_csv(f) for f in fichiers_csv]
df_global = pd.concat(liste_dfs, ignore_index=True)

# 4. Sauvegarder le résultat dans un unique fichier CSV consolidé
df_global.to_csv("cv_dataset_global.csv", index=False)

print(f"Concaténation réussie ! Total de lignes : {len(df_global)}")

15 fichiers trouvés.
Concaténation réussie ! Total de lignes : 15000


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42  # fixé une fois pour toutes -> même split pour les 3 approches

def charger_trustpilot(path="cv_dataset_global.csv"):
    df = pd.read_csv(path)

    # Gestion des noms de colonnes potentiels
    if "company" in df.columns:
        df = df.rename(columns={"company": "company"})
    else:
        df["entreprise"] = "inconnue"

    if "note" in df.columns and "rating" not in df.columns:
        df = df.rename(columns={"note": "rating"})

    if "commentaire" in df.columns and "content" not in df.columns:
        df = df.rename(columns={"commentaire": "content"})

    # Si la colonne 'id' n'existe pas, on la crée à partir de l'index
    if "id" not in df.columns:
        df["id"] = [f"tp_{i}" for i in range(len(df))]
    else:
        df["id"] = df["id"].astype(str)

    df["fichier"] = "trustpilot_scraping_rows"

    # Filtrer uniquement les colonnes disponibles pour éviter les erreurs
    cols_existantes = [c for c in ["id", "company", "rating", "content", "source" "fichier"] if c in df.columns]
    return df[cols_existantes]



def categorie(note):
    if note <= 2:
        return "négatif"
    elif note == 3:
        return "mitigé"
    return "positif"


# Chargement et fusion sécurisée des fichiers
df = pd.concat([charger_trustpilot()], ignore_index=True)

# Nettoyage minimal
df = df.dropna(subset=["content"])
df["content"] = df["content"].astype(str).str.strip()
df = df[df["content"].str.len() > 0].reset_index(drop=True)

df["categorie"] = df["rating"].apply(categorie)

print(f"Total avis exploitables : {len(df)}")
print("\nRépartition des catégories :")
print(df["categorie"].value_counts())

# Split stratifié
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df["categorie"]
)

print(f"\nTrain : {len(train_df)} avis  |  Test : {len(test_df)} avis")

train_df.to_csv("train.csv", index=False)
test_df.to_csv("test.csv", index=False)
print("\nFichiers sauvegardés avec succès : train.csv, test.csv")

Total avis exploitables : 13662

Répartition des catégories :
categorie
positif    10150
négatif     3120
mitigé       392
Name: count, dtype: int64

Train : 10929 avis  |  Test : 2733 avis

Fichiers sauvegardés avec succès : train.csv, test.csv


In [4]:
import numpy as np
import pandas as pd
import torch
from torch.nn import CrossEntropyLoss
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "cmarkea/distilcamembert-base"
ORDRE = ["négatif", "mitigé", "positif"]
label2id = {label: i for i, label in enumerate(ORDRE)}
id2label = {i: label for label, i in label2id.items()}

# 1. DONNÉES
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df["label"] = train_df["categorie"].map(label2id)
test_df["label"] = test_df["categorie"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["content"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df[["content", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["content", "label"]]).map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. MODÈLE
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

poids_classes = compute_class_weight(
    class_weight="balanced", classes=np.arange(3), y=train_df["label"].values
)
poids_classes = torch.tensor(poids_classes, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=poids_classes.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 3. ENTRAÎNEMENT
training_args = TrainingArguments(
    output_dir="./camembert_valorise",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    fp16=True,  # Activé si GPU T4 actif
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    # Ligne 'tokenizer=tokenizer' supprimée ici pour éviter l'erreur de type
    compute_metrics=compute_metrics,
)

trainer.train()

# 4. SAUVEGARDE DU MODÈLE CAMEMBERT (Format Hugging Face)
model_save_path = "modele_sentiment_camembert"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"\nModèle et tokenizer sauvegardés dans le dossier : {model_save_path}")

Map:   0%|          | 0/10929 [00:00<?, ? examples/s]

Map:   0%|          | 0/2733 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: cmarkea/distilcamembert-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.385903,0.808934,0.950238,0.690767
2,0.584823,0.719711,0.947677,0.731158
3,0.539829,0.846488,0.947311,0.736678


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Modèle et tokenizer sauvegardés dans le dossier : modele_sentiment_camembert


In [5]:
import shutil
shutil.make_archive("modele_sentiment_camembert", 'zip', "modele_sentiment_camembert")

'/content/modele_sentiment_camembert.zip'

In [3]:
import numpy as np
import pandas as pd
import torch
from torch.nn import CrossEntropyLoss
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "cmarkea/distilcamembert-base"
ORDRE = ["négatif", "mitigé", "positif"]
label2id = {label: i for i, label in enumerate(ORDRE)}
id2label = {i: label for label, i in label2id.items()}

# 1. DONNÉES
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
train_df["label"] = train_df["categorie"].map(label2id)
test_df["label"] = test_df["categorie"].map(label2id)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["content"], truncation=True, max_length=256)

train_ds = Dataset.from_pandas(train_df[["content", "label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["content", "label"]]).map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. MODÈLE
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

poids_classes = compute_class_weight(
    class_weight="balanced", classes=np.arange(3), y=train_df["label"].values
)
poids_classes = torch.tensor(poids_classes, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=poids_classes.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# 3. ENTRAÎNEMENT
training_args = TrainingArguments(
    output_dir="./camembert_valorise",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    fp16=True,  # Activé si GPU T4 actif
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

# 4. SAUVEGARDE DU MODÈLE CAMEMBERT (Format Hugging Face)
model_save_path = "modele_sentiment_camembert"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"\nModèle et tokenizer sauvegardés dans le dossier : {model_save_path}")

config.json:   0%|          | 0.00/732 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B /  811kB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/10929 [00:00<?, ? examples/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  273MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] CamembertForSequenceClassification LOAD REPORT from: cmarkea/distilcamembert-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: Trainer.__init__() got an unexpected keyword argument 'tokenizer'